## Решение TSP методами GA (DEAP)

### Импорты

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from functools import partial
from sklearn.model_selection import ParameterSampler
import plotly.graph_objects as go

### Генерация данных 2D

In [2]:
def generate_cities(n_cities: int):
    # Создаём N случайных городов на плоскости
    cities = np.random.rand(n_cities, 2) * 100  # координаты [x, y]

    # Матрица расстояний (евклидово расстояние)
    dist_matrix = np.zeros((n_cities, n_cities))
    for i in range(n_cities):
        for j in range(n_cities):
            dist_matrix[i][j] = np.linalg.norm(cities[i] - cities[j])
    return cities, dist_matrix

### Функция оптимизации (длина маршрута)

In [3]:
def evalTSP(individual, dist_matrix):
    # individual – список городов в порядке обхода
    distance = 0
    for i in range(len(individual)):
        from_city = individual[i]
        to_city = individual[(i+1) % len(individual)]  # замыкаем маршрут
        distance += dist_matrix[from_city][to_city]
    return (distance,) # кортеж из одного элемента

### Генетический алгоритм

In [4]:
def optimize_tsp(cities, dist_matrix, pop_size=300, ngen=200, cxpb=0.8, mutpb=0.2, mutindpb=0.05, tournsize=3):
    """_summary_

    Args:
        cities (_type_): _description_
        dist_matrix (_type_): _description_
        pop_size (_type_): размер популяции
        ngen (_type_): количество поколений
        cxpb (_type_): вероятность кроссовера
        mutpb (_type_): вероятность мутации
        mutindpb (_type_): вероятность мутации каждого отдельного элемента в особи
        tournsize (_type_): размер турнира

    Returns:
        _type_: _description_
    """
    n_cities = len(cities)

    # Создаём классы: максимизация приспособленности (но мы будем минимизировать,
    # поэтому используем weights=(-1.0,)) – чем меньше расстояние, тем лучше.
    if not hasattr(creator, "FitnessMin"):
        creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
    if not hasattr(creator, "Individual"):
        creator.create('Individual', list, fitness=creator.FitnessMin) # особь = список городов

    toolbox = base.Toolbox()

    # Генератор случайной перестановки городов
    toolbox.register("indices", random.sample, range(n_cities), n_cities)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    toolbox.register("evaluate", partial(evalTSP, dist_matrix=dist_matrix))

    # Для перестановок используем упорядоченный кроссовер (OX)
    toolbox.register("mate", tools.cxOrdered)
    # Мутация: инверсия случайного сегмента
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=mutindpb)
    # Отбор: турнирный (размер турнира 3)
    toolbox.register("select", tools.selTournament, tournsize=tournsize)

    # Создаём начальную популяцию
    pop = toolbox.population(n=pop_size)

    # Статистика для наблюдения за процессом
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Используем простой алгоритм eaSimple (с элитизмом можно добавить)
    pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=cxpb, mutpb=mutpb,
                                    ngen=ngen, stats=stats, verbose=True)

    return pop, logbook

### Запуск эволюции

In [5]:
cities, dist_matrix = generate_cities(n_cities=100)

In [15]:
param_grid = {
    'pop_size': [300, 400, 500],
    'ngen': [300, 400, 500],
    'cxpb': [0.3, 0.6, 0.9],
    'mutpb': [0.3, 0.6, 0.9],
    'mutindpb': [0.05, 0.1, 0.3],
    'tournsize': [3, 5, 10, 15],
}

In [18]:
grid_log = {'params': [], 'pops': [], 'logbooks': []}
grid_best = None
grid_best_dist = None
for params in ParameterSampler(param_grid, n_iter=20, random_state=42):
    print(params)
    pop, logbook = optimize_tsp(cities=cities, dist_matrix=dist_matrix, **params)
    grid_log['params'].append(params)
    grid_log['pops'].append(pop)
    grid_log['logbooks'].append(logbook)
    if grid_best:
        curr_best = tools.selBest(pop, 1)[0]
        curr_best_dist = evalTSP(curr_best, dist_matrix)[0]
        if grid_best_dist > curr_best_dist:
            grid_best = curr_best
            grid_best_dist = curr_best_dist
    else:
        grid_best = tools.selBest(pop, 1)[0]
        grid_best_dist = evalTSP(grid_best, dist_matrix)[0]
print(f"\nЛучший маршрут: {grid_best}")
print(f"Длина лучшего маршрута: {grid_best_dist:.3f}")


{'tournsize': 15, 'pop_size': 500, 'ngen': 300, 'mutpb': 0.3, 'mutindpb': 0.1, 'cxpb': 0.9}
gen	nevals	avg    	min    	max    
0  	500   	5276.72	4650.26	5856.84
1  	476   	5009.16	4370.94	5604.65
2  	465   	4807.76	4307.78	5513.08
3  	464   	4656   	4193.78	5264.69
4  	460   	4544.94	4074.08	5264.58
5  	467   	4441.68	3965.48	5166.3 
6  	478   	4335.29	3904.86	5098.93
7  	460   	4276.31	3685.74	5263.15
8  	472   	4170.7 	3697.5 	5099.91
9  	459   	4094.7 	3627.31	4972.14
10 	467   	4043.79	3575.03	4998.75
11 	460   	3966.86	3464.97	4972.82
12 	472   	3893.27	3402.29	4877.34
13 	476   	3821.42	3382.07	4888.45
14 	456   	3773.7 	3311.07	4984.15
15 	469   	3689.95	3139.27	4766.81
16 	453   	3673.61	3139.27	4954.35
17 	465   	3563.81	3119.25	4708.38
18 	466   	3484.75	3097.36	4850.51
19 	458   	3422.25	3058.09	4421.81
20 	478   	3375.79	3026.71	4625.22
21 	456   	3338.87	2975.97	4762.38
22 	472   	3338.14	2915.47	4657.07
23 	463   	3269.51	2864.45	4433.77
24 	468   	3265.1 	2860.13	4659.4

### Визуализация

In [34]:
route_cities = cities[grid_best]
route_cities = np.vstack([route_cities, route_cities[0]])  # замыкаем маршрут
city_labels = [str(city) for city in grid_best] + [str(grid_best[0])]
fig_1 = go.Figure()
fig_1.add_trace(
    go.Scatter(
        x=route_cities[:, 0], y=route_cities[:, 1],
        mode='lines+markers',
        name='Маршрут',
        text=city_labels,
        textposition='top center',
        line=dict(color='blue', width=2),
        marker=dict(size=6, color='blue')
    )
)
fig_1.add_trace(
    go.Scatter(
        x=[route_cities[0, 0]], y=[route_cities[0, 1]],
        mode='markers',
        name='Старт',
        marker=dict(size=12, color='red')
    )
)
fig_1.update_layout(
    title=f'Оптимальный маршрут, длина = {grid_best_dist:.2f}',
    xaxis_title='X координата',
    yaxis_title='Y координата',
    width=800, height=600,
    showlegend=True
)
fig_1.show()

In [24]:
num_traces = len(grid_log['params'])
fig_2 = go.Figure()
for num_trace in range(num_traces):
    x = np.arange(len(grid_log['logbooks'][num_trace]))
    y = grid_log['logbooks'][num_trace].select('min')
    fig_2.add_trace(
        go.Scatter(x=x, y=y, mode='lines', name=f'GA {grid_log['params'][num_trace]}', hovertemplate='<b>%{fullData.name}</b><br>Поколение: %{x}<br>Длина: %{y:.2f}<extra></extra>')
    )
fig_2.update_layout(
    title=f'Длина маршрутов генетических алгоритмов для {len(cities)} городов',
    xaxis_title = 'Поколение',
    yaxis_title = 'Длина маршрута'
)
fig_2.show()